# 笔记本 10 — 完整管线集成

**第四阶段 · 评估与部署（2 / 2）**

---

## 🎯 学习目标

| # | 目标 |
|---|------|
| 1 | 将所有模块串联为一个**端到端交易循环** |
| 2 | 理解生产级 `TradingBot` 生命周期：引导 → 轮询 → 策略周期 |
| 3 | 模拟多天循环：数据 → 信号 → 制度 → 集成 → 组合 → 风控 → 执行 |
| 4 | 实现**状态持久化**（JSON 保存/加载） |
| 5 | 对照**生产架构**，与我们的演练进行比较 |

### 前置要求
- NB01–NB09（所有前序笔记本）

In [ ]:
# ── 环境设置 ──────────────────────────────────────────────
import sys, pathlib, warnings, json, math
from datetime import datetime, timezone
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ 导入完成  |  项目根目录:", ROOT)

---
## 1 · 生产架构概览

生产级 `TradingBot`（`main.py`）协调六个层级：

```
┌─────────────────────────────────────────────────────────────┐
│                      TradingBot                            │
│                                                             │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌────────────┐ │
│  │   数据    │→│   信号   │→│   策略    │→│   组合     │ │
│  │ Binance  │  │ 动量     │  │ 制度检测  │  │ 归一化     │ │
│  │ OHLCV    │  │ 均值回归 │  │ 集成策略  │  │ 优化       │ │
│  │ 行情轮询  │  │ 配对     │  │ 情绪叠加  │  │            │ │
│  └──────────┘  │ 板块     │  └──────────┘  └────────────┘ │
│                └──────────┘        │              │        │
│                                    ▼              ▼        │
│  ┌────────────┐  ┌───────────────────────────────────────┐ │
│  │   监控      │  │           风控 & 执行                 │ │
│  │ Telegram    │  │  熔断器 → 仓位限制 → 订单生成 → 提交   │ │
│  │ 指标追踪    │  │                                       │ │
│  └────────────┘  └───────────────────────────────────────┘ │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  调度器: 轮询 (60秒) → 策略 (300秒) → 心跳            │   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

### 关键生命周期事件

| 事件 | 方法 | 频率 |
|------|------|------|
| **引导** | `bootstrap()` | 启动时1次 |
| **行情轮询** | `run_poll_cycle()` | 每60秒 |
| **策略周期** | `run_operational_cycle()` | 每300秒 |
| **心跳** | `send_heartbeat()` | 每3600秒 |
| **时钟同步** | `sync_server_time()` | 每3600秒 |

---
## 2 · 合成数据生成

In [ ]:
np.random.seed(42)
N_DAYS = 120
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=N_DAYS, freq="D")

assets = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
          "ADAUSDT", "AVAXUSDT", "DOTUSDT"]

# Generate correlated returns with regime structure
market = np.random.normal(0.001, 0.015, N_DAYS)
# Inject a bear regime in the middle
market[40:60] -= 0.012
# Inject a bull regime later
market[80:100] += 0.008

prices = pd.DataFrame(index=dates)
base_prices = {"BTCUSDT": 60000, "ETHUSDT": 3500, "SOLUSDT": 150, "BNBUSDT": 600,
               "XRPUSDT": 0.60, "ADAUSDT": 0.45, "AVAXUSDT": 35, "DOTUSDT": 7}

for i, asset in enumerate(assets):
    beta = 0.6 + 0.4 * i / len(assets)
    alpha = 0.0003 * (len(assets) - i) / len(assets)
    idio = np.random.normal(0, 0.012, N_DAYS)
    r = alpha + beta * market + idio
    prices[asset] = base_prices[asset] * np.exp(np.cumsum(r))

# Volume (synthetic)
volumes = pd.DataFrame(
    np.random.lognormal(mean=15, sigma=1.5, size=(N_DAYS, len(assets))),
    index=dates, columns=assets
)

print(f"价格面板: {prices.shape[0]} 天 × {prices.shape[1]} 资产")
print(f"制度结构: 熊市 @ 第40-60天, 牛市 @ 第80-100天")
prices.tail(3)

---
## 3 · 逐级构建管线各阶段

我们将每个模块实现为独立函数，然后链式串联。

In [ ]:
# ── 阶段1: 制度检测 ──────────────────────────────────────

def detect_regime(prices_btc: pd.Series, ema_short: int = 20, ema_long: int = 50,
                  vol_lookback: int = 14, vol_multiplier: float = 1.5) -> str:
    """将当前市场制度分类为 bull（牛市）、bear（熊市）或 ranging（震荡）。"""
    if len(prices_btc) < ema_long + 1:
        return "ranging"
    ema_s = prices_btc.ewm(span=ema_short, adjust=False).mean()
    ema_l = prices_btc.ewm(span=ema_long, adjust=False).mean()
    returns = prices_btc.pct_change().dropna()
    recent_vol = returns.iloc[-vol_lookback:].std()
    baseline_vol = returns.std()
    trending = ema_s.iloc[-1] > ema_l.iloc[-1]
    high_vol = recent_vol > baseline_vol * vol_multiplier
    if trending and not high_vol:
        return "bull"
    elif not trending and high_vol:
        return "bear"
    return "ranging"

# Test
regime = detect_regime(prices["BTCUSDT"])
print(f"当前制度: {regime}")

In [ ]:
# ── 阶段2: 信号生成 ──────────────────────────────────────

def momentum_signal(prices: pd.DataFrame, lookbacks: list = [3, 5, 7],
                    top_n: int = 5) -> dict[str, float]:
    """按多回看期动量排名，返回权重映射。"""
    scores = pd.Series(0.0, index=prices.columns)
    for lb in lookbacks:
        ret = prices.pct_change(lb).iloc[-1]
        norm = (ret - ret.min()) / (ret.max() - ret.min() + 1e-10)
        scores += norm
    scores /= len(lookbacks)
    top = scores.nlargest(top_n)
    total = top.sum()
    if total == 0:
        return {sym: 1.0 / top_n for sym in top.index}
    return {sym: float(w / total) for sym, w in top.items()}

def mean_reversion_signal(prices: pd.DataFrame, rsi_period: int = 14,
                          rsi_oversold: float = 30) -> dict[str, float]:
    """通过 RSI 寻找超卖资产，返回权重映射。"""
    weights = {}
    for asset in prices.columns:
        delta = prices[asset].diff()
        gain = delta.clip(lower=0).rolling(rsi_period).mean()
        loss = (-delta.clip(upper=0)).rolling(rsi_period).mean()
        rs = gain / (loss + 1e-10)
        rsi = 100 - (100 / (1 + rs))
        current_rsi = rsi.iloc[-1]
        if current_rsi < rsi_oversold:
            weights[asset] = float(1.0 - current_rsi / 100.0)
    if weights:
        total = sum(weights.values())
        return {k: v / total for k, v in weights.items()}
    return {}

# Generate signals
mom_weights = momentum_signal(prices)
mr_weights = mean_reversion_signal(prices)
print(f"动量选出 {len(mom_weights)} 个资产: {list(mom_weights.keys())}")
print(f"均值回归选出 {len(mr_weights)} 个资产: {list(mr_weights.keys())}")

In [ ]:
# ── 阶段3: 集成融合 ──────────────────────────────────────

REGIME_WEIGHTS = {
    "bull":    {"momentum": 0.50, "mean_reversion": 0.10, "sentiment": 0.20, "sector": 0.20},
    "ranging": {"momentum": 0.20, "mean_reversion": 0.50, "sentiment": 0.30, "sector": 0.00},
    "bear":    {"momentum": 0.00, "mean_reversion": 0.30, "sentiment": 0.20, "sector": 0.00},
}

CASH_FLOORS = {"bull": 0.20, "ranging": 0.40, "bear": 0.50}

def ensemble_combine(regime: str, signal_maps: dict[str, dict]) -> dict[str, float]:
    """使用制度依赖乘数融合子策略权重映射。"""
    blend = REGIME_WEIGHTS.get(regime, REGIME_WEIGHTS["ranging"])
    combined = {}
    for strategy_name, weight_map in signal_maps.items():
        factor = blend.get(strategy_name, 0.0)
        for sym, w in weight_map.items():
            combined[sym] = combined.get(sym, 0.0) + w * factor
    return combined

raw_ensemble = ensemble_combine(regime, {
    "momentum": mom_weights,
    "mean_reversion": mr_weights,
})
print(f"制度: {regime}")
print(f"原始集成权重（{len(raw_ensemble)} 个资产）:")
for sym, w in sorted(raw_ensemble.items(), key=lambda x: -x[1]):
    print(f"  {sym:12s} {w:.4f}")

In [ ]:
# ── 阶段4: 组合归一化 ────────────────────────────────────

def normalize_weights(weights: dict[str, float], cash_floor: float) -> dict[str, float]:
    """将正权重缩放使其总和为 (1 - cash_floor)。"""
    positives = {k: v for k, v in weights.items() if v > 0}
    if not positives:
        return {}
    total = sum(positives.values())
    target_sum = 1.0 - cash_floor
    return {k: v / total * target_sum for k, v in positives.items()}

cash_floor = CASH_FLOORS[regime]
normalized = normalize_weights(raw_ensemble, cash_floor)

print(f"现金底线 ({regime}): {cash_floor:.0%}")
print(f"已投资:  {sum(normalized.values()):.2%}")
print(f"现金:    {1 - sum(normalized.values()):.2%}")
for sym, w in sorted(normalized.items(), key=lambda x: -x[1]):
    print(f"  {sym:12s} {w:.4f}")

In [ ]:
# ── 阶段5: 风险控制 ──────────────────────────────────────

def enforce_position_limit(weights: dict[str, float], max_position: float = 0.10) -> dict[str, float]:
    """将每个仓位裁剪到 max_position。"""
    return {k: min(v, max_position) for k, v in weights.items()}

def check_circuit_breaker(drawdown: float, l1: float = 0.03, l2: float = 0.05) -> str:
    """返回 'halt'（停止）、'reduce'（减仓）或 'ok'。"""
    if abs(drawdown) >= l2:
        return "halt"
    elif abs(drawdown) >= l1:
        return "reduce"
    return "ok"

risk_adjusted = enforce_position_limit(normalized, max_position=0.10)
print(f"仓位限制后（最大10%）:")
for sym, w in sorted(risk_adjusted.items(), key=lambda x: -x[1]):
    print(f"  {sym:12s} {w:.4f}")

In [ ]:
# ── 阶段6: 订单生成 ──────────────────────────────────────

def generate_rebalance_orders(current_weights: dict[str, float],
                               target_weights: dict[str, float],
                               min_drift: float = 0.15) -> list[dict]:
    """为偏离超过阈值的权重生成 BUY/SELL 订单。"""
    all_symbols = set(list(current_weights.keys()) + list(target_weights.keys()))
    orders = []
    for sym in sorted(all_symbols):
        curr = current_weights.get(sym, 0.0)
        tgt = target_weights.get(sym, 0.0)
        drift = abs(tgt - curr) / max(curr, 1e-10)
        if drift > min_drift:
            side = "BUY" if tgt > curr else "SELL"
            orders.append({"side": side, "symbol": sym, "target_weight": tgt, "drift": drift})
    return orders

# Start from no positions
current_positions = {asset: 0.0 for asset in assets}
orders = generate_rebalance_orders(current_positions, risk_adjusted)

print(f"生成 {len(orders)} 个订单:")
for o in orders:
    print(f"  {o['side']:4s} {o['symbol']:12s} 目标权重={o['target_weight']:.4f}")

---
## 4 · 状态持久化

生产机器人使用**原子写入**（写入临时文件，然后重命名）将状态保存到 `data/bot_state.json`。以下是简化版本：

In [ ]:
import tempfile, os

def save_state(state: dict, filepath: str = "/tmp/bot_state_demo.json") -> None:
    """原子保存：先写入临时文件，再重命名。"""
    directory = os.path.dirname(filepath)
    fd, tmp_path = tempfile.mkstemp(dir=directory, suffix=".tmp")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(state, f, indent=2, sort_keys=True, default=str)
            f.write("\n")
        os.replace(tmp_path, filepath)
    except:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
        raise

def load_state(filepath: str = "/tmp/bot_state_demo.json") -> dict:
    """从 JSON 文件加载状态。"""
    if not os.path.exists(filepath):
        return {}
    with open(filepath, "r") as f:
        return json.load(f)

# Demo state
demo_state = {
    "portfolio_value": 10000.0,
    "positions": current_positions,
    "regime": regime,
    "last_updated": datetime.now(timezone.utc).isoformat(),
}
save_state(demo_state)
loaded = load_state()
print(f"保存并加载状态: {list(loaded.keys())}")
print(f"组合价值: ${loaded['portfolio_value']:,.2f}")

---
## 5 · 完整管线模拟（多天循环）

现在将所有阶段串联成一个处理120天数据的**每日交易循环**：

In [ ]:
# ── 完整管线配置 ─────────────────────────────────────────
CONFIG = {
    "regime": {"ema_short": 20, "ema_long": 50, "vol_lookback": 14, "vol_multiplier": 1.5},
    "momentum": {"lookbacks": [3, 5, 7], "top_n": 5},
    "mean_reversion": {"rsi_period": 14, "rsi_oversold": 30},
    "risk": {"max_position": 0.10, "circuit_breaker_l1": 0.03, "circuit_breaker_l2": 0.05},
    "execution": {"min_drift": 0.15},
}

# ── 管线循环 ─────────────────────────────────────────────
WARMUP = max(CONFIG["regime"]["ema_long"], CONFIG["mean_reversion"]["rsi_period"]) + 5

portfolio_value = 10000.0
current_weights = {asset: 0.0 for asset in assets}
equity_curve = []
regime_history = []
order_log = []
circuit_breaker_log = []

peak_value = portfolio_value

for day in range(WARMUP, N_DAYS):
    date = dates[day]
    window = prices.iloc[:day + 1]
    
    # Stage 1: Regime
    reg = detect_regime(window["BTCUSDT"], **CONFIG["regime"])
    regime_history.append({"date": date, "regime": reg})
    
    # Stage 2: Signals
    mom_w = momentum_signal(window, **CONFIG["momentum"])
    mr_w = mean_reversion_signal(window, **CONFIG["mean_reversion"])
    
    # Stage 3: Ensemble
    raw = ensemble_combine(reg, {"momentum": mom_w, "mean_reversion": mr_w})
    
    # Stage 4: Normalize
    cf = CASH_FLOORS[reg]
    norm_w = normalize_weights(raw, cf)
    
    # Stage 5: Risk
    drawdown = (portfolio_value / peak_value) - 1 if peak_value > 0 else 0
    cb_status = check_circuit_breaker(drawdown, **{k.replace("circuit_breaker_", ""): v 
                                                    for k, v in CONFIG["risk"].items() 
                                                    if "circuit" in k})
    circuit_breaker_log.append({"date": date, "status": cb_status, "drawdown": drawdown})
    
    if cb_status == "halt":
        target_w = {asset: 0.0 for asset in assets}
    elif cb_status == "reduce":
        target_w = {k: v * 0.5 for k, v in norm_w.items()}
    else:
        target_w = enforce_position_limit(norm_w, CONFIG["risk"]["max_position"])
    
    # Stage 6: Orders
    day_orders = generate_rebalance_orders(current_weights, target_w, CONFIG["execution"]["min_drift"])
    if day_orders:
        order_log.append({"date": date, "count": len(day_orders), "regime": reg})
    
    # Apply weights (simulate)
    current_weights = target_w
    
    # Compute portfolio return for this day
    if day < N_DAYS - 1:
        daily_ret = prices.iloc[day + 1] / prices.iloc[day] - 1
        port_return = sum(current_weights.get(a, 0) * daily_ret.get(a, 0) for a in assets)
        portfolio_value *= (1 + port_return)
        peak_value = max(peak_value, portfolio_value)
    
    equity_curve.append({"date": date, "value": portfolio_value, "regime": reg})

eq_df = pd.DataFrame(equity_curve).set_index("date")
print(f"模拟完成: {len(equity_curve)} 天")
print(f"最终组合价值: ${portfolio_value:,.2f}  ({(portfolio_value/10000 - 1)*100:+.2f}%)")
print(f"再平衡事件: {len(order_log)} 次")

---
## 6 · 管线可视化

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1, 1]})

# ── 面板1: 资金曲线与制度背景 ────────────────────────────
ax1 = axes[0]
ax1.plot(eq_df.index, eq_df["value"], lw=1.5, color="#2c3e50", label="投资组合")

# Buy & hold benchmark
bench_ret = prices.iloc[WARMUP:].pct_change().mean(axis=1)
bench_eq = 10000 * (1 + bench_ret).cumprod()
ax1.plot(bench_eq.index, bench_eq.values, lw=1.2, color="#95a5a6", ls="--", label="等权持有基准")

# Regime shading
colors = {"bull": "#2ecc71", "ranging": "#f1c40f", "bear": "#e74c3c"}
for i in range(len(eq_df) - 1):
    ax1.axvspan(eq_df.index[i], eq_df.index[i+1], alpha=0.08,
                color=colors.get(eq_df["regime"].iloc[i], "gray"))

ax1.set_ylabel("组合价值 ($)")
ax1.set_title("完整管线模拟")
patches = [mpatches.Patch(color=c, alpha=0.3, label=r) for r, c in colors.items()]
ax1.legend(handles=[*ax1.get_legend_handles_labels()[0][:2], *patches], loc="upper left", fontsize=8)

# ── 面板2: 回撤 ─────────────────────────────────────────
ax2 = axes[1]
equity_series = eq_df["value"]
peak = equity_series.cummax()
dd = (equity_series / peak) - 1
ax2.fill_between(dd.index, dd, alpha=0.4, color="#e74c3c")
ax2.axhline(-0.03, ls="--", color="orange", lw=0.8, label="L1 (-3%)")
ax2.axhline(-0.05, ls="--", color="red", lw=0.8, label="L2 (-5%)")
ax2.set_ylabel("回撤")
ax2.legend(fontsize=8)
ax2.invert_yaxis()

# ── 面板3: 再平衡活动 ───────────────────────────────────
ax3 = axes[2]
if order_log:
    ol_df = pd.DataFrame(order_log)
    bar_colors = [colors.get(r, "gray") for r in ol_df["regime"]]
    ax3.bar(ol_df["date"], ol_df["count"], color=bar_colors, alpha=0.7, width=1)
ax3.set_ylabel("订单数")
ax3.set_xlabel("日期")

plt.tight_layout()
plt.show()

---
## 7 · 管线统计

In [ ]:
regime_counts = pd.DataFrame(regime_history)["regime"].value_counts()
cb_df = pd.DataFrame(circuit_breaker_log)
cb_counts = cb_df["status"].value_counts()

print("═" * 50)
print(" 管线总结")
print("═" * 50)
print(f"\n  模拟周期:    {eq_df.index[0].date()} → {eq_df.index[-1].date()}")
print(f"  交易天数:    {len(equity_curve)}")
print(f"  初始资金:    $10,000.00")
print(f"  最终价值:    ${portfolio_value:,.2f}")
print(f"  总收益率:    {(portfolio_value/10000 - 1)*100:+.2f}%")
print(f"  最大回撤:    {dd.min():.2%}")

print(f"\n  制度分布:")
for r, count in regime_counts.items():
    print(f"    {r:10s} {count:4d} 天 ({count/len(regime_history)*100:.1f}%)")

print(f"\n  熔断器触发:")
for status, count in cb_counts.items():
    print(f"    {status:10s} {count:4d} 天")

print(f"\n  再平衡事件:  {len(order_log)} 次")
if order_log:
    total_orders = sum(o["count"] for o in order_log)
    print(f"  总订单数:    {total_orders}")

---
## 8 · 与生产代码对照

我们的模拟与生产级 `TradingBot` 的对应关系：

| 我们的模拟 | 生产代码 | 文件 |
|-----------|---------|------|
| `detect_regime()` | `detect_regime()` | `bot/strategy/regime_detector.py` |
| `momentum_signal()` | `rank_assets_by_momentum()` | `bot/signals/momentum.py` |
| `mean_reversion_signal()` | `find_oversold_assets()` | `bot/signals/mean_reversion.py` |
| `ensemble_combine()` | `ensemble_combine()` | `bot/strategy/ensemble.py` |
| `normalize_weights()` | `normalize_weights()` | `bot/strategy/portfolio_optimizer.py` |
| `enforce_position_limit()` | `enforce_position_limit()` | `bot/risk/risk_manager.py` |
| `check_circuit_breaker()` | `CircuitBreaker.evaluate()` | `bot/risk/circuit_breaker.py` |
| `generate_rebalance_orders()` | `generate_rebalance_orders()` | `bot/execution/order_executor.py` |
| `save_state()` / `load_state()` | `TradingBot.save_state()` / `load_state()` | `bot/main.py` |
| 每日循环 | `APScheduler`（60秒轮询 / 300秒策略） | `bot/main.py` |

### 生产级额外功能

| 功能 | 说明 |
|------|------|
| **Roostoo API** | 通过 REST API 执行真实订单 |
| **Telegram 告警** | 心跳 + 交易通知 |
| **时钟同步** | 服务器时间偏移，确保时间戳准确 |
| **原子状态写入** | `tempfile.mkstemp()` + `os.replace()` |
| **APScheduler** | 类 Cron 作业调度 |
| **配对轮换** | 基于协整的配对交易 |
| **板块轮换** | 基于 BTC 市值占比的配置 |
| **情绪叠加** | 恐惧贪婪指数乘数 |

In [ ]:
# Quick check: import production classes to verify accessibility
from bot.main import TradingBot, StrategyCycleResult
from bot.strategy.ensemble import ensemble_combine as prod_ensemble, _REGIME_WEIGHTS
from bot.strategy.portfolio_optimizer import normalize_weights as prod_normalize
from bot.risk.risk_manager import enforce_position_limit as prod_position_limit
from bot.risk.circuit_breaker import CircuitBreaker
from bot.execution.order_executor import generate_rebalance_orders as prod_orders

print("生产级导入成功 ✅")
print(f"\n生产级制度权重:")
for regime_name, weights in _REGIME_WEIGHTS.items():
    formatted = ", ".join(f"{k}={v:.2f}" for k, v in weights.items())
    print(f"  {regime_name:8s}: {formatted}")

# Verify circuit breaker
cb = CircuitBreaker()
for dd_val in [0.01, 0.03, 0.05, 0.08]:
    print(f"  CB({dd_val:.0%} DD) → {cb.evaluate(dd_val)}")

---
## 9 · 权重随时间的演变

In [ ]:
# Re-run pipeline to capture weight history
weight_history = []
for day in range(WARMUP, N_DAYS):
    window = prices.iloc[:day + 1]
    reg = detect_regime(window["BTCUSDT"], **CONFIG["regime"])
    mom_w = momentum_signal(window, **CONFIG["momentum"])
    mr_w = mean_reversion_signal(window, **CONFIG["mean_reversion"])
    raw = ensemble_combine(reg, {"momentum": mom_w, "mean_reversion": mr_w})
    norm_w = normalize_weights(raw, CASH_FLOORS[reg])
    risk_w = enforce_position_limit(norm_w, CONFIG["risk"]["max_position"])
    
    row = {"date": dates[day]}
    row.update({a: risk_w.get(a, 0.0) for a in assets})
    row["cash"] = max(0, 1.0 - sum(risk_w.values()))
    weight_history.append(row)

wh_df = pd.DataFrame(weight_history).set_index("date")

fig, ax = plt.subplots(figsize=(14, 6))
cols = assets + ["cash"]
cmap = plt.cm.Set3(np.linspace(0, 1, len(cols)))
ax.stackplot(wh_df.index, *[wh_df[c] for c in cols], labels=cols, colors=cmap, alpha=0.8)
ax.set_ylabel("权重")
ax.set_xlabel("日期")
ax.set_title("投资组合权重演变")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 10 · 核心要点

| 概念 | 说明 |
|------|------|
| **六层架构** | 数据 → 信号 → 策略 → 组合 → 风控 → 执行 |
| **制度自适应** | 权重根据牛市/震荡/熊市检测动态调整 |
| **现金底线** | 20%（牛市）、40%（震荡）、50%（熊市） |
| **熔断器** | L1（3%回撤）→ 减仓，L2（5%回撤）→ 停止 |
| **状态持久化** | 原子 JSON 写入，用于崩溃恢复 |
| **调度器** | APScheduler 运行轮询（60秒）和策略（300秒）作业 |

### 本课程构建的完整体系

| 笔记本 | 主题 | 核心技能 |
|--------|------|----------|
| NB00 | 课程总览 | 架构、学习路径 |
| NB01 | 市场数据 | API 调用、OHLCV、SQLite |
| NB02 | 技术指标 | EMA、RSI、布林带 |
| NB03 | 风险指标 | 夏普、索提诺、VaR、回撤 |
| NB04 | 动量策略 | 多回看期评分、排名 |
| NB05 | 均值回归与配对 | BB 信号、协整检验 |
| NB06 | 制度检测 | EMA 交叉、波动率制度 |
| NB07 | 集成与情绪 | 制度权重、恐贪指数、板块轮换 |
| NB08 | 组合与风控 | 归一化、仓位限制、熔断器 |
| NB09 | 回测引擎 | 滚动前推、指标计算、参数扫描 |
| **NB10** | **完整管线** | **端到端集成、状态持久化** |

---
## 🔬 练习

1. **添加情绪叠加：** 生成合成的恐惧贪婪时间序列，用情绪乘数（限制在 [0.5, 1.5]）调整集成权重。

2. **交易成本：** 加入每次再平衡 10bp 的往返成本，衡量对最终组合价值的影响。

3. **动态资产池：** 从4个资产开始，在第60天加入2个新资产。管线如何处理新进资产？

4. **Telegram 模拟：** 创建一个模拟告警器，打印格式化消息（启动、心跳、熔断器触发、每日摘要）。

5. **实盘数据扩展：** 使用 NB01 的 `BinanceFetcher` 替换合成数据为真实 Binance 数据，运行完整管线。

---
## ✅ 知识检查

1. 交易管线的六个层级按顺序是什么？
2. 制度如何影响现金底线百分比？
3. 熔断器达到 L2 状态时会发生什么？
4. 为什么生产机器人使用原子写入来持久化状态？
5. 生产调度器运行策略周期和行情轮询的频率分别是多少？

---
## 🎓 课程完成！

恭喜——你已经从零构建了一个**制度自适应、多策略加密货币交易系统**。

### 后续步骤

- 阅读 `bot/` 目录下的生产源代码，学习生产级设计模式
- 运行回测：`python -m bot.main backtest --symbols BTCUSDT,ETHUSDT`
- 深入研究 `Technicals/06_Strategy_Mathematics_Deep_Dive.md` 了解完整数学推导
- 查阅 `docs/03_operations_runbook.md` 了解部署说明

祝交易顺利！🚀